# Erwin ParticleTransformer — Local Smoke Test

Runs the **Erwin (BallMSA) attention** model through weaver's *real* data pipeline on a
single JetClass ROOT file, so you can confirm it works before spending cluster time.

**What it checks**
1. The ROOT file loads and preprocesses through weaver's `SimpleIterDataset` (same as training).
2. The Erwin model builds via `get_model` and takes the Erwin (`ball` attention) path — no pair embedding.
3. A forward pass gives valid logits (right shape, no NaN/Inf).
4. A short train loop drives the **loss down** and **accuracy up**, i.e. gradients flow through BallMSA.

**Prerequisites**
- Run this notebook with the `weaver` conda env kernel (`conda activate weaver`).
- Edit the paths in the **Config** cell below if yours differ.

## 1. Config — edit these if needed

In [20]:
import os

# --- paths (edit if yours differ) ---
REPO_DIR       = os.path.expanduser('~/Documents/HEP-NN/efficient_particle_transformer')
ROOT_FILE      = os.path.expanduser('~/Documents/HEP-NN/JetClass_example_100k.root')
DATA_CONFIG    = os.path.join(REPO_DIR, 'data/JetClass/JetClass_full.yaml')
NETWORK_CONFIG = os.path.join(REPO_DIR, 'networks/example_ErwinParticleTransformer.py')

# --- smoke-test knobs ---
DEVICE     = 'cpu'   # CPU is safest; some BallMSA ops may be unsupported on MPS
BATCH_SIZE = 100
N_STEPS    = 200      # number of gradient steps to run

# Optional: shrink the model for a faster test (default {} = faithful to get_model).
# e.g. MODEL_OVERRIDES = dict(num_layers=2, embed_dims=[64, 128, 64], num_heads=4)
MODEL_OVERRIDES = {}

for p in (REPO_DIR, ROOT_FILE, DATA_CONFIG, NETWORK_CONFIG):
    assert os.path.exists(p), f'Path not found: {p}'
print('All paths exist.')

All paths exist.


## 2. Fix the `particle_transformer` import path

The network wrapper does `from particle_transformer.networks...`, but this repo is checked out
as `efficient_particle_transformer`. We make `particle_transformer` resolve to this repo via a
symlink on `sys.path` (no changes to your real directories).

> On the cluster, make sure the repo is checked out (or symlinked) as `particle_transformer`, or the same import will fail there.

In [21]:
import sys, os

_pyroot = os.path.expanduser('~/.cache/part_smoke_pyroot')
os.makedirs(_pyroot, exist_ok=True)
_link = os.path.join(_pyroot, 'particle_transformer')
if not (os.path.islink(_link) and os.readlink(_link) == REPO_DIR):
    if os.path.lexists(_link):
        os.remove(_link)
    os.symlink(REPO_DIR, _link)
if _pyroot not in sys.path:
    sys.path.insert(0, _pyroot)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)   # so 'networks/...', 'data/...' relative paths resolve

import particle_transformer  # noqa: F401  -> should import cleanly now
print('particle_transformer ->', os.path.realpath(particle_transformer.__path__[0]))

particle_transformer -> /Users/vaibhavlohiya/Documents/HEP-NN/efficient_particle_transformer


## 3. Inspect the ROOT file

In [22]:
import uproot

with uproot.open(ROOT_FILE) as f:
    tree = f[f.keys()[0]]
    print('tree:', tree.name, '| entries:', tree.num_entries)
    part_branches  = [k for k in tree.keys() if k.startswith('part_')]
    label_branches = [k for k in tree.keys() if k.startswith('label_')]
    print('particle branches:', part_branches)
    print('label branches   :', label_branches)

tree: tree | entries: 100000
particle branches: ['part_px', 'part_py', 'part_pz', 'part_energy', 'part_deta', 'part_dphi', 'part_d0val', 'part_d0err', 'part_dzval', 'part_dzerr', 'part_charge', 'part_isChargedHadron', 'part_isNeutralHadron', 'part_isPhoton', 'part_isElectron', 'part_isMuon']
label branches   : ['label_QCD', 'label_Hbb', 'label_Hcc', 'label_Hgg', 'label_H4q', 'label_Hqql', 'label_Zqq', 'label_Wqq', 'label_Tbqq', 'label_Tbl']


## 4. Build weaver's data pipeline

`SimpleIterDataset` applies the exact preprocessing defined in the YAML (standardization,
padding to length 128, feature stacking) — the same code path training uses.

In [23]:
import torch
from weaver.utils.dataset import SimpleIterDataset

file_dict = {'_': [ROOT_FILE]}
dataset = SimpleIterDataset(
    file_dict, DATA_CONFIG, for_training=True,
    fetch_by_files=True, fetch_step=1, in_memory=True, name='smoke',
)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)
data_config = dataset.config
print('input groups :', data_config.input_names)
print('label name   :', data_config.label_names[0])
print('num classes  :', len(data_config.label_value))
print('pf_features dim:', len(data_config.input_dicts['pf_features']))

input groups : ('pf_points', 'pf_features', 'pf_vectors', 'pf_mask')
label name   : _label_
num classes  : 10
pf_features dim: 17


## 5. Peek at one batch

In [24]:
batch_iter = iter(loader)
X, y, Z = next(batch_iter)
for k in data_config.input_names:
    print(f'{k:12s}', tuple(X[k].shape))
labels = y[data_config.label_names[0]].long()
print('labels shape  :', tuple(labels.shape))
print('class counts  :', torch.bincount(labels, minlength=len(data_config.label_value)).tolist())

=== Restarting DataIter smoke, seed=None ===
pf_points    (100, 2, 128)
pf_features  (100, 17, 128)
pf_vectors   (100, 4, 128)
pf_mask      (100, 1, 128)
labels shape  : (100,)
class counts  : [6, 12, 10, 11, 11, 11, 10, 11, 5, 13]


## 6. Build the Erwin model

We load `get_model` straight from the network-config file (as weaver does) and assert we're on
the Erwin path: `pair_embed is None` and `use_erwin_blocks is True`.

In [25]:
import importlib.util

spec = importlib.util.spec_from_file_location('erwin_net_cfg', NETWORK_CONFIG)
net_cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(net_cfg)

model, model_info = net_cfg.get_model(data_config, **MODEL_OVERRIDES)
model = model.to(DEVICE)
loss_fn = net_cfg.get_loss(data_config)

n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

core = model.mod   # EfficientParticleTransformer inside the wrapper
assert core.use_erwin_blocks, 'Expected Erwin blocks!'
assert core.pair_embed is None, 'Erwin path should have no pair embedding!'
print('Confirmed: Erwin (BallMSA) path, no pair embedding.')

parameters: 2,652,062
Confirmed: Erwin (BallMSA) path, no pair embedding.


## 7. Forward-pass sanity check

In [26]:
model.eval()
with torch.no_grad():
    inputs = [X[k].to(DEVICE) for k in data_config.input_names]
    logits = model(*inputs)

n_classes = len(data_config.label_value)
assert logits.shape == (labels.shape[0], n_classes), logits.shape
assert torch.isfinite(logits).all(), 'Non-finite values in logits!'
probs = torch.softmax(logits, dim=1)
print('logits shape :', tuple(logits.shape))
print('all finite   :', bool(torch.isfinite(logits).all()))
print('softmax row-sum ~1:', torch.allclose(probs.sum(1), torch.ones(probs.shape[0]), atol=1e-5))

logits shape : (100, 10)
all finite   : True
softmax row-sum ~1: True


## 8. Short training loop

Drive a handful of gradient steps and watch the loss. This proves gradients flow through BallMSA.
On this tiny, unbalanced single file you won't get real accuracy — you just want the loss to trend
**down** from ~ln(10)≈2.30 and accuracy to lift off the ~0.1 random-chance floor.

In [27]:
import itertools

model.train()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

losses, accs = [], []
for step, (X, y, Z) in zip(range(N_STEPS), itertools.cycle(loader)):
    inputs = [X[k].to(DEVICE) for k in data_config.input_names]
    label  = y[data_config.label_names[0]].long().to(DEVICE)

    opt.zero_grad()
    logits = model(*inputs)
    loss = loss_fn(logits, label)
    loss.backward()
    opt.step()

    acc = (logits.argmax(1) == label).float().mean().item()
    losses.append(loss.item()); accs.append(acc)
    if step % 5 == 0 or step == N_STEPS - 1:
        print(f'step {step:3d}  loss {loss.item():.4f}  acc {acc:.3f}')

print(f'\nfirst-5 avg loss {sum(losses[:5])/5:.4f}  ->  last-5 avg loss {sum(losses[-5:])/5:.4f}')

step   0  loss 2.3530  acc 0.150
step   5  loss 2.3530  acc 0.060
step  10  loss 2.3493  acc 0.070
step  15  loss 2.3103  acc 0.230
step  20  loss 2.2940  acc 0.140
step  25  loss 2.2982  acc 0.130
step  30  loss 2.3001  acc 0.130
step  35  loss 2.2709  acc 0.110
step  40  loss 2.2279  acc 0.130
step  45  loss 2.2854  acc 0.140
step  50  loss 2.0966  acc 0.190
step  55  loss 1.9988  acc 0.250
step  60  loss 2.0200  acc 0.270
step  65  loss 2.1303  acc 0.220
step  70  loss 2.0668  acc 0.240
step  75  loss 2.1130  acc 0.240
step  80  loss 2.1368  acc 0.190
step  85  loss 2.1882  acc 0.170
step  90  loss 1.8630  acc 0.300
step  95  loss 1.9305  acc 0.280
step 100  loss 2.0543  acc 0.260
step 105  loss 1.7831  acc 0.340
step 110  loss 1.7854  acc 0.280
step 115  loss 1.5751  acc 0.340
step 120  loss 1.8349  acc 0.350
step 125  loss 1.5315  acc 0.350
step 130  loss 1.5490  acc 0.420
step 135  loss 1.5605  acc 0.470
step 140  loss 1.4696  acc 0.450
step 145  loss 1.5317  acc 0.420
step 150  

## 9. Plot the curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses, marker='o', ms=3)
ax[0].axhline(2.302, ls='--', c='gray', label='ln(10) random')
ax[0].set_title('Training loss'); ax[0].set_xlabel('step'); ax[0].set_ylabel('loss'); ax[0].legend()
ax[1].plot(accs, marker='o', ms=3, c='tab:green')
ax[1].axhline(0.1, ls='--', c='gray', label='random (1/10)')
ax[1].set_title('Training accuracy'); ax[1].set_xlabel('step'); ax[1].set_ylabel('acc'); ax[1].legend()
plt.tight_layout()

# Save the loss/accuracy figure next to this notebook, then display it.
_out = os.path.join(REPO_DIR, 'loss_accuracy_Erwin.png')
fig.savefig(_out, dpi=130, bbox_inches='tight')
print('saved ->', _out)
plt.show()

## What to look for

- **Cells 1-7 pass without error** -> the Erwin attention wiring, data pipeline, and forward pass are sound. This is the real smoke-test bar.
- **Loss trends downward** in cell 8/9 -> gradients are flowing through BallMSA and the model can fit.
- Don't read into absolute accuracy: this is one small, class-imbalanced file with only ~30 steps. Real evaluation needs the full dataset on the cluster.

**Next steps**
- Bump `N_STEPS` (e.g. 200) or set `MODEL_OVERRIDES` smaller for a quicker signal that the loss keeps dropping.
- When happy, launch the real run: `./train_JetClass.sh ErwinParT full` (ensure the repo is importable as `particle_transformer` on the cluster).